In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
 
from sklearn.metrics import mean_squared_error, r2_score 
from sklearn.neighbors import KNeighborsRegressor 
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor 
from sklearn.svm import SVR 
from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error 
from sklearn.model_selection import RandomizedSearchCV 
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor 
import warnings 
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../notebook/data/stud.csv")

In [3]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


### Preparando as features X e Y

In [4]:
X = df.drop(columns=["math_score"], axis=1)

In [5]:
y = df['math_score']

### Vamos transformar colunas categóricas em numéricas

In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer 

num_features = X.select_dtypes(exclude=["object"]).columns
cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler() 
oh_transformer = OneHotEncoder() 

preprocessor = ColumnTransformer( 
    [
        ("OneHotEncoder", oh_transformer, cat_features), 
        ("StandardScaler", numeric_transformer, num_features) 
    ]
)




In [7]:
X = preprocessor.fit_transform(X) #Aplica a transformação dos dados aplicas pelo preprocessor

### Separar os dados em treino e teste

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)  #Toda vez que você executar essa célula, os dados serão divididos exatamente da mesma maneira. As mesmas linhas irão para o treino e as mesmas para o teste. Isso é crucial para que você possa comparar resultados.Se você mudar o modelo e a acurácia subir, você saberá que foi por causa da mudança no modelo, e não porque os dados de treino "deram sorte" de serem mais fáceis nessa execução.
X_train.shape, X_test.shape

((800, 19), (200, 19))

### Criar uma função para retornar as métricas do modelo

In [14]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    return mae, mse, rmse, r2
    

In [18]:

models = {
    "Linear Regression": LinearRegression(), 
    "Lasso": Lasso(), 
    "Ridge": Ridge(), 
    "K-Neighbors Regressor": KNeighborsRegressor(), 
    "Decision Tree": DecisionTreeRegressor(), 
    "Random Forest Regressor": RandomForestRegressor(), 
    "XGBRegressor": XGBRegressor(), 
    "CatBoostRegressor": CatBoostRegressor(verbose=False), 
    "AdaBoostRegressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # modelo de treinamento

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)  # fazendo a predição

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2  = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for training set')
    print("-  Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- Mean Squared Error: {:.4f}".format(model_train_mse))
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- R2 Score: {:.4f}".format(model_train_r2))
    print("")


    print('-----------------')
    print('Model performance for test set')
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- Mean Squared Error: {:.4f}".format(model_test_mse))
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    print('='*35)
    print('\n')





Linear Regression
Model performance for training set
-  Mean Absolute Error: 4.2667
- Mean Squared Error: 28.3349
- Root Mean Squared Error: 5.3231
- R2 Score: 0.8743

-----------------
Model performance for test set
- Mean Absolute Error: 4.2148
- Mean Squared Error: 29.0952
- Root Mean Squared Error: 5.3940
- R2 Score: 0.8804


Lasso
Model performance for training set
-  Mean Absolute Error: 5.2063
- Mean Squared Error: 43.4784
- Root Mean Squared Error: 6.5938
- R2 Score: 0.8071

-----------------
Model performance for test set
- Mean Absolute Error: 5.1579
- Mean Squared Error: 42.5064
- Root Mean Squared Error: 6.5197
- R2 Score: 0.8253


Ridge
Model performance for training set
-  Mean Absolute Error: 4.2650
- Mean Squared Error: 28.3378
- Root Mean Squared Error: 5.3233
- R2 Score: 0.8743

-----------------
Model performance for test set
- Mean Absolute Error: 4.2111
- Mean Squared Error: 29.0563
- Root Mean Squared Error: 5.3904
- R2 Score: 0.8806


K-Neighbors Regressor
Model 